# config

In [ ]:
config = {
    "file_config": {
        "output_path": "s3a://web-parse-hw60p/renpengli/agentic_search/",
    },
    "db_config": {
        "starrocks": {
            "host": "",
            "port": 30030,
            "user": "",
            "password": ""
        }
    },
    "custom_config": {
        "spark_profile": "test",
        "starrocks_query_timeout_seconds": 1800,
        "sql": "",
        "iceberg_path": "s3://lakehouse-iceberg/",
        "detail_s3_paths": ["s3://lakehouse/", "s3://lakehouse2/"],
    }
}


# 准备 OA chunk 输入数据

只取 **OA** 数据（canonical per-sha 口径：同一 sha 只要有一行 OA 即视为 OA），
把 **metadata 表字段** 与 **真实 doc 字段**（`access_xinghe_repository_processed_path` 指向的 jsonl）
合并成一份 DataFrame，供后续 chunk 处理使用。

- 同名字段（`doi`/`title`/`author`/`language`/`abstract`）只保留一份，`coalesce(真实 doc, metadata)`。
- **不包含 `middle_json`**：真实 doc 的 schema 由样本 jsonl 推断（`infer_real_doc_schema`），
  推断后排除 `middle_json`（`EXCLUDE_REAL_FIELDS`），按该 schema 读取即不含它。
- 本 notebook **只产出 DataFrame**（`oa_chunk_df`），不写盘。


In [ ]:
import os
os.environ["SPARK_USER"] = "renpengli"
os.environ.setdefault("XINGHE_CONF_DIR", "/share/renpengli/conf")

# Spark 配置：本 notebook = 读 StarRocks metadata + 读 OA 真实 jsonl + join + CPU 密集的 chunk 切分。
# 取向：足够并发读 S3 小文件；分区数“确定且足够多”，让 chunk 的 mapPartitions 切得细、抗长尾 doc。
import bz2
import gzip
import io
import json
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from pyspark import SparkConf, StorageLevel
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, MapType, ArrayType, LongType

from xinghe.s3 import *
from xinghe.spark import *
from xinghe.utils.json_util import *

from admin_ingest.env_bootstrap import load_dotenvs
from admin_ingest.settings import Settings
from admin_ingest.storage.chunk_time_iso import utc_millis_to_iso_z
from admin_ingest.chunk.ingest_fields import INGEST_EXTRA_KEYS
from admin_ingest.spark_bridge import RecursiveChunkOptions, chunk_plain_from_dict_row

# config = json_loads('${config}')
custom_config = config["custom_config"]
file_config = config["file_config"]
s3_config = config.get("s3_config", {})
db_config = config.get("db_config", {})

base_output_path = file_config["output_path"]
starrocks_config = db_config["starrocks"]
spark_profile = custom_config.get("spark_profile", "test")  # 验证用 test，全量改 "prod"
current_dt = datetime.now().strftime('%Y-%m-%d')

def get_s3_args(path, s3_config):
    bucket = split_s3_path(path)[0]
    s3_args = s3_config.get(bucket, {})
    if not s3_args:
        s3_args = get_s3_config(path)
    print(f"bucket={bucket}, s3_args={s3_args}")
    return bucket, s3_args
    

detail_s3_paths = custom_config.get("detail_s3_paths", ["s3://lakehouse/"])
detail_s3_bucket_configs = {}
for detail_s3_path in detail_s3_paths:
    detail_s3_bucket, detail_s3_config = get_s3_args(detail_s3_path, s3_config)
    detail_s3_bucket_configs[detail_s3_bucket] = detail_s3_config

write_s3_bucket, write_s3_config = get_s3_args(base_output_path, s3_config)

iceberg_path = custom_config["iceberg_path"]
write_iceberg_bucket, write_iceberg_config = get_s3_args(iceberg_path, s3_config)

STARROCKS_CONNECTION = {
    **starrocks_config,
    "database": "ads",
    "charset": "utf8mb4",
}

METADATA_BASE_SQL = custom_config.get("sql", "").strip()
if not METADATA_BASE_SQL:
    raise ValueError("custom_config.sql is required")
STARROCKS_JDBC_QUERY_TIMEOUT_SECONDS = custom_config.get("starrocks_query_timeout_seconds", 1800)
STARROCKS_PATH_QUERY_TIMEOUT_SECONDS = custom_config.get("starrocks_query_timeout_seconds", 1800)

if spark_profile == "prod":
    QUERY_LIMIT = None
    JDBC_NUM_PARTITIONS = 256
    JDBC_FETCH_SIZE = 10000
    SCHEMA_SAMPLE_FILES = 200
    SOURCE_FILE_BATCH_SIZE = 2000
    WRITE_PARTITIONS = 1000
    PREVIEW_ROWS = 5
    STARROCKS_PATH_QUERY_BATCH_SIZE = 2000
    CHUNK_OUTPUT_ROOT = f"{base_output_path.rstrip('/')}/oa_chunk_data/{current_dt}"
else:
    QUERY_LIMIT = 50
    JDBC_NUM_PARTITIONS = 8
    JDBC_FETCH_SIZE = 1000
    SCHEMA_SAMPLE_FILES = 20
    SOURCE_FILE_BATCH_SIZE = 10
    WRITE_PARTITIONS = 8
    PREVIEW_ROWS = 5
    STARROCKS_PATH_QUERY_BATCH_SIZE = 10
    CHUNK_OUTPUT_ROOT = f"{base_output_path.rstrip('/')}/oa_chunk_data_test/{current_dt}_test"
    
# chunk 结果写出位置：
# - CHUNK_OUTPUT_ROOT: 汇总目录（_SUMMARY.json 等）
# - CHUNK_DATA_ROOT: 实际 chunk 数据，按 batch 子目录分桶，便于断点续跑
# - SOURCE_PATH_PROGRESS_ROOT: 已完成 source_jsonl_path checkpoint
CHUNK_DATA_ROOT = f"{CHUNK_OUTPUT_ROOT.rstrip('/')}/data"
SOURCE_PATH_PROGRESS_ROOT = f"{CHUNK_OUTPUT_ROOT.rstrip('/')}/_source_path_progress".replace("s3://", "s3a://", 1)
CHUNK_WRITE_MODE = "overwrite"  # 每批写独立 batch 目录；重跑同批覆盖该 batch，避免重复。
ERROR_OUTPUT_ROOT = f"{CHUNK_OUTPUT_ROOT.rstrip('/')}/error"

SPARK_JOB_ID = "spark-app-001"
CHUNK_INPUT_EXTRA_INFO_MAX_BYTES = 65536
# 分批生成 chunk：按 source_jsonl_path checkpoint 断点续跑；每批写到独立 batch 目录，重跑仅覆盖当前批次。
OVERWRITE_EXISTING_CHUNKS = False
INTRA_BATCH_OVERSIZED_BYTES_THRESHOLD = 2 * 1024 * 1024
INTRA_BATCH_TARGET_ROWS_PER_PARTITION = 8

# metadata 表需要的源列（字段合集里的「metadata 表字段」）。
METADATA_TABLE_COLUMNS = [
    "metadata_type",
    "doi",
    "isbn13",
    "title",
    "author",
    "language",
    "access_xinghe_repository_origin_url",
    "access_xinghe_repository_origin_path",
    "access_xinghe_repository_processed_path",
    "access_xinghe_repository_sha256",
    "access_is_oa",
    "abstract",
    "publication_published_date",
    "publication_published_year",
    "publication_venue_type",
    "publication_venue_name_unified",
    "primary_topic",
    "citation_count",
    "influential_citation_count",
    "dt",
]

# config 里的 SQL 统一映射回 notebook 内部列名。
METADATA_SQL_SELECT_CLAUSE = ",\n             ".join([
    "metadata_type AS metadata_type",
    "doi AS doi",
    "isbn13 AS isbn13",
    "title AS title",
    "author AS author",
    "language AS language",
    "origin_url AS access_xinghe_repository_origin_url",
    "origin_path AS access_xinghe_repository_origin_path",
    "processed_path AS access_xinghe_repository_processed_path",
    "source_sha256 AS access_xinghe_repository_sha256",
    "access_is_oa AS access_is_oa",
    "abstract AS abstract",
    "publication_published_date AS publication_published_date",
    "publication_published_year AS publication_published_year",
    "publication_venue_type AS publication_venue_type",
    "publication_venue_name_unified AS publication_venue_name_unified",
    "primary_topic AS primary_topic",
    "citation_count AS citation_count",
    "influential_citation_count AS influential_citation_count",
    "dt AS dt",
])

# metadata 结果按此 schema 对齐 DataFrame（全部按字符串处理）。
RAW_METADATA_SCHEMA = T.StructType([
    T.StructField(name, T.StringType(), True) for name in METADATA_TABLE_COLUMNS
])

# 字段合集输出时，metadata 表字段的「输出列名 -> oa_metadata_df 内部列名」。
# 仅 access_xinghe_repository_sha256 内部叫 source_sha256，其余同名。
METADATA_OUTPUT_FIELDS = {
    "metadata_type": "metadata_type",
    "doi": "doi",
    "isbn13": "isbn13",
    "title": "title",
    "author": "author",
    "language": "language",
    "access_xinghe_repository_origin_url": "access_xinghe_repository_origin_url",
    "access_xinghe_repository_origin_path": "access_xinghe_repository_origin_path",
    "access_xinghe_repository_processed_path": "access_xinghe_repository_processed_path",
    "access_xinghe_repository_sha256": "source_sha256",
    "access_is_oa": "access_is_oa",
    "abstract": "abstract",
    "publication_published_date": "publication_published_date",
    "publication_published_year": "publication_published_year",
    "publication_venue_type": "publication_venue_type",
    "publication_venue_name_unified": "publication_venue_name_unified",
    "primary_topic": "primary_topic",
    "citation_count": "citation_count",
    "influential_citation_count": "influential_citation_count",
}

# 真实 doc 字段不再硬编码，而是对样本 jsonl 做 schema 推断后排除这些重字段。
# middle_json 是 MinerU 中间结构，极大且 chunk 用不到。
EXCLUDE_REAL_FIELDS = {"middle_json"}

# chunk 最终写出字段：以 md 中最终口径为准，去掉用户明确不要的 query / entity_keys / embedding。
FINAL_CHUNK_FIELDS = [
    "chunk_id",
    "doc_id",
    "title",
    "content",
    "abstract",
    "url",
    "category",
    "source_type",
    "lang",
    "job_id",
    "task_id",
    "extra_info",
    "created_time",
    "offset",
    "page_no",
    "chunk_no",
    "publication_published_date",
    "publication_published_year",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "primary_topic_subfield",
    "primary_topic_field",
    "primary_topic_domain",
    "author",
    "publication_venue_name_unified",
    "access_is_oa",
    "citation_count",
    "influential_citation_count",
    "updated_time",
]

# chunk 断点续跑：按 source_jsonl_path 记录 checkpoint；每批写到独立 batch 目录，失败重跑仅覆盖当前批次。
SOURCE_PATH_PROGRESS_SCHEMA = T.StructType([
    T.StructField("source_jsonl_path", T.StringType(), True),
    T.StructField("batch_index", T.IntegerType(), True),
    T.StructField("batch_doc_count", T.LongType(), True),
    T.StructField("batch_chunk_count", T.LongType(), True),
    T.StructField("completed_at", T.StringType(), True),
])

CHUNK_CALL_ROW_KEYS = {
    "doc_id",
    "sha256",
    "content",
    "text",
    "content_list",
    "page_no",
    "language",
    "lang",
    "source_type",
    "category",
    "job_id",
    "task_id",
}
CHUNK_INPUT_CORE_KEYS = {
    "doc_id",
    "title",
    "url",
    "origin_url",
    "origin_path",
    "language",
    "lang",
    "content",
    "text",
    "content_list",
    "source",
    "source_type",
    "category",
    "job_id",
    "task_id",
    "doc_loc",
    "processed_path",
    "sha256",
    "extra_info",
    "abstract",
    "publication_published_date",
    "publication_published_year",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "publication_venue_name_unified",
    "access_is_oa",
    "citation_count",
    "influential_citation_count",
    "author",
    "doi",
    "isbn13",
    "isbn",
    "issn",
    "source_jsonl_path",
    "source_sha256",
    "source_has_bytes_suffix",
    "access_xinghe_repository_processed_path",
}
CHUNK_INPUT_ROW_KEYS = set(INGEST_EXTRA_KEYS) | CHUNK_INPUT_CORE_KEYS
ERROR_ROW_KEYS = {
    "doc_id",
    "title",
    "url",
    "origin_url",
    "origin_path",
    "language",
    "lang",
    "source",
    "source_type",
    "category",
    "job_id",
    "task_id",
    "doc_loc",
    "processed_path",
    "sha256",
    "abstract",
    "publication_published_date",
    "publication_published_year",
    "metadata_type",
    "publication_venue_type",
    "primary_topic",
    "publication_venue_name_unified",
    "access_is_oa",
    "citation_count",
    "influential_citation_count",
    "author",
    "doi",
    "isbn13",
    "isbn",
    "issn",
    "source_jsonl_path",
    "source_sha256",
    "source_has_bytes_suffix",
    "access_xinghe_repository_processed_path",
}


In [ ]:
base_config = {
    # spark 其他配置
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.speculation": "true",
    "spark.executorEnv.SPARK_USER": "renpengli",
    # hadoop s3a 配置
    "spark.hadoop.fs.s3a.impl":"org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    # "spark.hadoop.fs.s3a.path.style.access": "true",
    # "spark.hadoop.fs.s3a.endpoint": write_s3_config["endpoint"],
    # "spark.hadoop.fs.s3a.access.key": write_s3_config["ak"],
    # "spark.hadoop.fs.s3a.secret.key": write_s3_config["sk"],
    # ========== 桶1：agentic_search 独立认证与endpoint ==========
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.endpoint": write_s3_config["endpoint"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.access.key": write_s3_config["ak"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.secret.key": write_s3_config["sk"],
    f"spark.hadoop.fs.s3a.bucket.{write_s3_bucket}.path.style.access": "true",
    "spark.hadoop.fs.s3a.multiobjectdelete.enable":"false",
    "spark.hadoop.fs.s3a.directory.marker.retention":"keep",
    "spark.hadoop.fs.s3a.fast.upload":"true",
    "spark.hadoop.fs.s3a.connection.maximum":"1000",
    "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version":"2",
    # kubernetes 配置
    "spark.kubernetes.executor.deleteOnTermination":"false",
    "spark.submit.deployMode": "cluster",
    "spark.kubernetes.namespace": "dataops-paas",
    "spark.kubernetes.authenticate.driver.serviceAccountName": "spark-default",
    "spark.kubernetes.container.image.pullPolicy": "Always",
    "spark.kubernetes.container.image": "registry.sensetime.com/hadoop/dataops/apache/spark:3.5.7-data-platform",
    # iceberg 配置
    "spark.sql.defaultCatalog": "iceberg-dataops",
    "spark.sql.catalog.iceberg-dataops.uri": "thrift://pjlab-dataproducer-hive-metastore.pjlab-dataproducer.svc.cluster.local:9083",
    "spark.sql.catalog.iceberg-dataops.warehouse": "s3a://lakehouse-iceberg/",
    "spark.sql.catalog.iceberg-dataops": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.iceberg-dataops.type": "hive",
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.access.key": write_iceberg_config["ak"],
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.secret.key": write_iceberg_config["sk"],
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.endpoint": write_iceberg_config["endpoint"],
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.fast.upload": "true",
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.path.style.access": "true",
    "spark.sql.catalog.iceberg-dataops.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",  # 创建bloom索引必须得参数，需iceberg版本大于2.0
    # event log 配置
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": "/spark/eventLog",
    # Magic Committer（推荐 - 不需要临时目录，性能最佳）
    # 参考: https://hadoop.apache.org/docs/current/hadoop-aws/tools/hadoop-aws/committers.html
    "spark.jars":"/share/spark-jars/spark-hadoop-cloud_2.12-3.5.7.jar",
    "spark.hadoop.fs.s3a.committer.name": "magic",
    "spark.hadoop.fs.s3a.committer.magic.enabled": "true" ,
    "spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a": "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory",
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter",
    "spark.kubernetes.executor.podTemplateFile": "/share/renpengli/pod-template/pod-template-dolphin.yaml",
    "spark.kubernetes.executor.label.queue": "root.datacenter.data-producer.default",
    "spark.driver.host": os.environ.get("POD_IP"),
    "spark.submit.deployMode": "client",
}   

for detail_s3_bucket, detail_s3_config in detail_s3_bucket_configs.items():
    base_config.update({
        f"spark.hadoop.fs.s3a.bucket.{detail_s3_bucket}.endpoint": detail_s3_config["endpoint"],
        f"spark.hadoop.fs.s3a.bucket.{detail_s3_bucket}.access.key": detail_s3_config["ak"],
        f"spark.hadoop.fs.s3a.bucket.{detail_s3_bucket}.secret.key": detail_s3_config["sk"],
        f"spark.hadoop.fs.s3a.bucket.{detail_s3_bucket}.path.style.access": "true",
    })
    
if spark_profile == "prod":
    spark_config = {
        **base_config,
        # spark 资源配置
        "spark.driver.memory": "8g",
        "spark.driver.maxResultSize": "4g",
        "spark.executor.memory": "12g",
        "spark.executor.memoryOverhead": "4g",
        "spark.executor.cores": "4",
        "spark.executor.instances": "200",
        "spark.network.timeout": "800s",
        "spark.kubernetes.executor.limit.cores": "8",  # spark.kubernetes.executor.limit.cores 是对cpu的限制
        "spark.default.parallelism": "4000",
        #sparksql 配置
        "spark.sql.sources.partitionOverwriteMode":"dynamic",
        "spark.sql.parquet.compression.codec":"snappy",
        "spark.sql.files.maxRecordsPerFile": 50000,
        "spark.sql.adaptive.enabled":"true",
        "spark.sql.shuffle.partitions":"4096",
        "spark.sql.adaptive.coalescePartitions.enabled":"true",
        "spark.sql.adaptive.advisoryPartitionSizeInBytes":"256MB",
        "spark.sql.autoBroadcastJoinThreshold":"-1",
        "spark.sql.adaptive.shuffle.targetPostShuffleInputSize":"67108864",
        "spark.dynamicAllocation.minExecutors": "50",
        "spark.dynamicAllocation.initialExecutors": "100",
        "spark.dynamicAllocation.maxExecutors": "400",
    }
else:
    spark_config = {
        **base_config,
        # spark 资源配置
        "spark.driver.memory": "4g",
        "spark.driver.maxResultSize": "2g",
        "spark.executor.memory": "4g",
        "spark.executor.memoryOverhead": "2g",
        "spark.executor.cores": "4",
        "spark.executor.instances": "50",
        "spark.network.timeout": "800s",
        "spark.kubernetes.executor.limit.cores": "8",  # spark.kubernetes.executor.limit.cores 是对cpu的限制
        "spark.default.parallelism": "64",
        #sparksql 配置
        "spark.sql.sources.partitionOverwriteMode":"dynamic",
        "spark.sql.parquet.compression.codec":"snappy",
        "spark.sql.files.maxRecordsPerFile": 50000,
        "spark.sql.adaptive.enabled":"true",
        "spark.sql.shuffle.partitions":"64",
        "spark.sql.adaptive.coalescePartitions.enabled":"true",
        "spark.sql.adaptive.advisoryPartitionSizeInBytes":"256MB",
        "spark.sql.autoBroadcastJoinThreshold":"-1",
        "spark.sql.adaptive.shuffle.targetPostShuffleInputSize":"67108864",
        "spark.dynamicAllocation.minExecutors": "2",
        "spark.dynamicAllocation.initialExecutors": "4",
        "spark.dynamicAllocation.maxExecutors": "16",
    }

conf = SparkConf() 
conf.setAll(list(spark_config.items()))

# 初始化Spark
master = "k8s://https://{kubernetes_service_host}:{kubernetes_service_port}".format(kubernetes_service_host = os.environ.get("KUBERNETES_SERVICE_HOST"), 
        kubernetes_service_port = os.environ.get("KUBERNETES_SERVICE_PORT"))
tt = int(time.time())
spark = SparkSession.builder.master(master).config(conf=conf).appName(f"raw_to_chunk_{tt}").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

_jvm = spark._jvm
_hconf = spark._jsc.hadoopConfiguration()
_Path = _jvm.org.apache.hadoop.fs.Path

spark


In [ ]:
print(f"run profile: {spark_profile}")
print(f"QUERY_LIMIT={QUERY_LIMIT}")
print(f"JDBC_NUM_PARTITIONS={JDBC_NUM_PARTITIONS}")
print(f"SCHEMA_SAMPLE_FILES={SCHEMA_SAMPLE_FILES}")
print(f"SOURCE_FILE_BATCH_SIZE={SOURCE_FILE_BATCH_SIZE}")
print(f"STARROCKS_JDBC_QUERY_TIMEOUT_SECONDS={STARROCKS_JDBC_QUERY_TIMEOUT_SECONDS}")
print(f"STARROCKS_PATH_QUERY_TIMEOUT_SECONDS={STARROCKS_PATH_QUERY_TIMEOUT_SECONDS}")
print(f"STARROCKS_PATH_QUERY_BATCH_SIZE={STARROCKS_PATH_QUERY_BATCH_SIZE}")
print(f"EXCLUDE_REAL_FIELDS={sorted(EXCLUDE_REAL_FIELDS)}")
print(f"FINAL_CHUNK_FIELDS={FINAL_CHUNK_FIELDS}")
print(f"CHUNK_OUTPUT_ROOT={CHUNK_OUTPUT_ROOT}")
print(f"CHUNK_DATA_ROOT={CHUNK_DATA_ROOT}")
print(f"SOURCE_PATH_PROGRESS_ROOT={SOURCE_PATH_PROGRESS_ROOT}")
print(f"CHUNK_WRITE_MODE={CHUNK_WRITE_MODE}, WRITE_PARTITIONS={WRITE_PARTITIONS}")
print(f"PREVIEW_ROWS={PREVIEW_ROWS}")

# funcs

In [ ]:
# ---- 查询 StarRocks + OA canonical 判定（本 notebook 自有的精简实现）----
def iter_value_chunks(values, chunk_size):
    step = max(1, int(chunk_size))
    for start in range(0, len(values), step):
        yield values[start : start + step]


def ensure_s3a_path(path):
    if path.startswith("s3://"):
        return "s3a://" + path[len("s3://"):]
    return path


def has_bytes_suffix_expr(column):
    return column.rlike(r"\?bytes=.*$")


def is_oa_expr(column):
    # access_is_oa 规整：大小写/空白无关，等于 'true' 才算 OA。
    return F.lower(F.trim(F.coalesce(column.cast("string"), F.lit("")))) == F.lit("true")


def with_metadata_dedup_columns(df):
    return (
        df
        .withColumn("doc_loc", F.trim(F.col("access_xinghe_repository_processed_path")))
        .where(F.col("doc_loc").isNotNull() & (F.col("doc_loc") != ""))
        .withColumn("source_jsonl_path", F.col("doc_loc"))
        .withColumn("source_sha256", F.trim(F.col("access_xinghe_repository_sha256")))
        .withColumn("source_has_bytes_suffix", has_bytes_suffix_expr(F.col("doc_loc")))
        .withColumn("raw_is_oa", is_oa_expr(F.col("access_is_oa")))
        .withColumn("metadata_dt", F.to_date(F.col("dt")))
    )


def dedupe_metadata_by_sha256(df):
    # 只保留有 sha256 的 metadata，并按 sha256 去重：优先最新 dt，其次优先 ?bytes= 路径。
    sha_df = df.where(F.col("source_sha256").isNotNull() & (F.col("source_sha256") != ""))
    if sha_df.rdd.isEmpty():
        return sha_df
    dedup_window = Window.partitionBy("source_sha256").orderBy(
        F.col("metadata_dt").desc_nulls_last(),
        F.col("source_has_bytes_suffix").cast("int").desc(),
    )
    dedup_sha_df = (
        sha_df
        .withColumn("_sha_rank", F.row_number().over(dedup_window))
        .where(F.col("_sha_rank") == 1)
        .drop("_sha_rank")
    )
    return dedup_sha_df


def build_metadata_sql(limit=None, oa_only=False, processed_paths=None, order_by_processed_path=True):
    # 统一从 config 里的基础 SQL 取数，再标准化成 notebook 内部使用的列名。
    where_clauses = []
    if oa_only:
        where_clauses.append("lower(trim(coalesce(access_is_oa, ''))) = 'true'")
    if processed_paths:
        quoted_paths = ", ".join(json.dumps(str(path)) for path in processed_paths)
        where_clauses.append(f"processed_path IN ({quoted_paths})")
    where_sql = ""
    if where_clauses:
        where_sql = "WHERE " + "\n          AND ".join(where_clauses)
    order_sql = "ORDER BY source_sha256, processed_path" if order_by_processed_path else ""
    limit_sql = "" if limit is None else f"LIMIT {int(limit)}"
    return f"""
        SELECT {METADATA_SQL_SELECT_CLAUSE}
        FROM ({METADATA_BASE_SQL}) metadata_base
        {where_sql}
        {order_sql}
        {limit_sql}
    """


def load_metadata_df(limit=None, oa_only=False, processed_paths=None, order_by_processed_path=True):
    sql = build_metadata_sql(
        limit=limit,
        oa_only=oa_only,
        processed_paths=processed_paths,
        order_by_processed_path=order_by_processed_path,
    )
    return spark.sql(sql)


def load_metadata_df_for_source_paths(source_paths):
    unique_paths = sorted({str(path).strip() for path in source_paths if path is not None and str(path).strip()})
    if not unique_paths:
        return spark.createDataFrame([], RAW_METADATA_SCHEMA)

    metadata_dfs = []
    total_chunks = (len(unique_paths) + STARROCKS_PATH_QUERY_BATCH_SIZE - 1) // STARROCKS_PATH_QUERY_BATCH_SIZE
    for chunk_index, path_chunk in enumerate(iter_value_chunks(unique_paths, STARROCKS_PATH_QUERY_BATCH_SIZE), start=1):
        chunk_df = load_metadata_df(processed_paths=path_chunk)
        metadata_dfs.append(chunk_df)
        print(
            f"[metadata][batch-query {chunk_index}/{total_chunks}] requested_paths={len(path_chunk)}"
        )

    merged_df = metadata_dfs[0]
    for chunk_df in metadata_dfs[1:]:
        merged_df = merged_df.unionByName(chunk_df)
    return merged_df


def load_batch_oa_metadata_df(source_paths):
    # 当前 batch 只回查这一批 source path 对应的 metadata，避免每个 batch 都重扫全量 oa_metadata_df。
    metadata_source_df = dedupe_metadata_by_sha256(with_metadata_dedup_columns(load_metadata_df_for_source_paths(source_paths)))
    return (
        metadata_source_df
        .select(
            F.col("metadata_type").alias("metadata_type"),
            F.col("doi").alias("doi"),
            F.col("isbn13").alias("isbn13"),
            F.col("title").alias("title"),
            F.col("author").alias("author"),
            F.col("language").alias("language"),
            F.col("access_xinghe_repository_origin_url").alias("access_xinghe_repository_origin_url"),
            F.col("access_xinghe_repository_origin_path").alias("access_xinghe_repository_origin_path"),
            F.col("access_xinghe_repository_processed_path").alias("access_xinghe_repository_processed_path"),
            F.col("abstract").alias("abstract"),
            F.col("publication_published_date").alias("publication_published_date"),
            F.col("publication_published_year").alias("publication_published_year"),
            F.col("publication_venue_type").alias("publication_venue_type"),
            F.col("publication_venue_name_unified").alias("publication_venue_name_unified"),
            F.col("primary_topic").alias("primary_topic"),
            F.col("citation_count").alias("citation_count"),
            F.col("influential_citation_count").alias("influential_citation_count"),
            F.col("source_sha256").alias("source_sha256"),
            F.col("doc_loc").alias("doc_loc"),
            F.col("source_jsonl_path").alias("source_jsonl_path"),
            F.col("source_has_bytes_suffix").alias("source_has_bytes_suffix"),
            # batch_paths 来自 canonical OA 的 source path 清单，所以这里统一回填为 'true'。
            F.lit("true").alias("access_is_oa"),
        )
    )


def infer_real_doc_schema(sample_paths, exclude=()):
    # 从样本 jsonl 推断真实 doc 的字段集合（含 content_list 的真实嵌套结构），
    # 再去掉 middle_json 等重字段。样本越多越能覆盖稀疏字段。
    if not sample_paths:
        raise ValueError("no sample paths to infer real doc schema")
    inferred = (
        spark.read
        .json([ensure_s3a_path(p) for p in sample_paths])
        .schema
    )
    exclude = set(exclude)
    # content_list 有三种源形态：对象数组 [{...}] / 被转义的 JSON 字符串 "[{...}]" / 纯字符串数组 ["..."]。
    # schema 推断只会把它定成其中一种(通常是对象数组)，另两种形态读进来就变成 null/丢正文。
    # 统一按 StringType 读成「原始 JSON 文本」，chunk 前再用 parse_content_list_value 还原，兼容三形态、不落 null
    # （与 startrocks_mineru_data.ipynb 的做法一致）。样本里若没出现 content_list，也补一个 StringType 占位列。
    fields = []
    seen_content_list = False
    for f in inferred.fields:
        if f.name in exclude:
            continue
        if f.name == "content_list":
            fields.append(T.StructField("content_list", T.StringType(), True))
            seen_content_list = True
        else:
            fields.append(f)
    if not seen_content_list and "content_list" not in exclude:
        fields.append(T.StructField("content_list", T.StringType(), True))
    return T.StructType(fields)

# ---- chunk 生成方法（显式组装最终写出字段）----
load_dotenvs()
settings = Settings.load()

def sanitize_chunk_extra_info(value):
    if value is None:
        return None
    payload = value
    if isinstance(payload, (bytes, bytearray)):
        try:
            payload = payload.decode("utf-8")
        except UnicodeDecodeError:
            return None
    if isinstance(payload, str):
        text = payload.strip()
        if not text:
            return None
        try:
            payload = json.loads(text)
        except (ValueError, TypeError):
            return None
    if not isinstance(payload, dict):
        return None
    try:
        encoded = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    except (TypeError, ValueError):
        return None
    if len(encoded) > CHUNK_INPUT_EXTRA_INFO_MAX_BYTES:
        return None
    return payload


def compact_chunk_input_row(row):
    compact = {}
    for key in CHUNK_INPUT_ROW_KEYS:
        if key not in row:
            continue
        value = row.get(key)
        if value is not None:
            compact[key] = value
    sanitized_extra_info = sanitize_chunk_extra_info(row.get("extra_info"))
    if sanitized_extra_info is not None:
        compact["extra_info"] = sanitized_extra_info
    else:
        compact.pop("extra_info", None)
    return compact


def build_chunk_call_row(row):
    # 只把 chunk_plain_from_dict_row 真正需要的字段传进去，避免它在 extra_info 中复制过大的原始 metadata。
    compact = {}
    for key in CHUNK_CALL_ROW_KEYS:
        if key not in row:
            continue
        value = row.get(key)
        if value is not None:
            compact[key] = value
    source_sha256 = normalize_sha256(row.get("source_sha256"))
    source_doc_id = str(row.get("doc_id") or "").strip()
    fallback_doc_id = source_doc_id or source_sha256 or str(row.get("doc_loc") or row.get("processed_path") or "").strip()
    if fallback_doc_id and not compact.get("doc_id"):
        compact["doc_id"] = fallback_doc_id
    if source_sha256 and not compact.get("sha256"):
        compact["sha256"] = source_sha256
    if not compact.get("language") and row.get("language") is not None:
        compact["language"] = row.get("language")
    if not compact.get("lang") and row.get("lang") is not None:
        compact["lang"] = row.get("lang")
    return compact


def compact_error_row(row):
    compact = {}
    for key in ERROR_ROW_KEYS:
        if key not in row:
            continue
        value = row.get(key)
        if value is not None:
            compact[key] = value
    content_list = row.get("content_list")
    if isinstance(content_list, list):
        compact["content_list_item_count"] = len(content_list)
    elif isinstance(content_list, str):
        compact["content_list_chars"] = len(content_list)
    elif content_list is not None:
        compact["content_list_type"] = type(content_list).__name__
    extra_info = sanitize_chunk_extra_info(row.get("extra_info"))
    if extra_info is not None:
        compact["extra_info"] = extra_info
    elif row.get("extra_info") is not None:
        compact["extra_info_dropped"] = True
    return compact


def parse_content_list_value(value):
    # content_list 兼容三种源数据形态，统一还原成 Python list/dict 再喂给 chunk，避免变空/null：
    #   1) 对象数组             [{"type":"text","text":"..."}, ...]
    #   2) 被转义的 JSON 字符串  "[{...}]"        (StringType 读入即为这段文本)
    #   3) 纯字符串数组          ["第一段", "第二段"]
    # real_doc_schema 已把 content_list 读成 StringType，这里拿到的恒为「一段 JSON 文本」或 None。
    # downstream chunk_plain_from_dict_row 只有在 isinstance(content_list, list) 时才走结构化 chunk；
    # 不还原成 list 的话三形态都会掉进平面 content/text 路径 -> 可能产出空 chunk 或丢正文。
    # 解析失败时原样返回字符串(不丢数据、便于排查)，而不是丢成 null。
    if value is None:
        return None
    if not isinstance(value, str):
        return value  # 兜底：上游已是结构化类型(list/dict)，直接用
    text = value.strip()
    if not text:
        return None
    try:
        parsed = json.loads(text)
    except (ValueError, TypeError):
        return value
    # 多层转义：解析后若仍是「像 JSON 容器的字符串」，继续解析直到拿到 list/dict。
    while isinstance(parsed, str):
        stripped = parsed.strip()
        if not (stripped.startswith("[") or stripped.startswith("{")):
            break
        try:
            parsed = json.loads(stripped)
        except (ValueError, TypeError):
            break
    return parsed


def filter_ref_text_content_list_items(value):
    # 当 content_list 是对象数组时，去掉引用文本节点，避免把 ref_text 一并送进 chunk。
    if not isinstance(value, list):
        return value
    if not value or not all(isinstance(item, dict) for item in value):
        return value
    return [
        item
        for item in value
        if item.get("type") != "ref_text" and item.get("sub_type") != "ref_text"
    ]


def build_chunk_error_record(row, error_reason, now_iso):
    return {
        "record_type": "error",
        "error_reason": error_reason,
        "error_time": now_iso,
        "doc_loc": row.get("doc_loc"),
        "source_jsonl_path": row.get("source_jsonl_path"),
        "source_sha256": row.get("access_xinghe_repository_sha256") or row.get("source_sha256") or row.get("sha256"),
        "metadata_type": row.get("metadata_type"),
        "title": row.get("title"),
        "raw_row": compact_error_row(row),
    }


def normalize_sha256(value):
    text = str(value or "").strip().lower()
    return text or None


def extract_doc_sha256_candidates(doc):
    candidates = []
    for key in ("sha256", "doc_id", "id"):
        value = doc.get(key)
        if value is None:
            continue
        text = str(value).strip().lower()
        if len(text) == 64 and all(ch in "0123456789abcdef" for ch in text):
            candidates.append(text)
    return list(dict.fromkeys(candidates))


def normalize_real_doc_for_chunk(doc, processed_path, row_loc=None):
    normalized = dict(doc)
    normalized["content_list"] = parse_content_list_value(normalized.get("content_list"))
    normalized["content_list"] = filter_ref_text_content_list_items(normalized.get("content_list"))
    normalized["doc_loc"] = normalized.get("doc_loc") or row_loc or processed_path
    normalized["processed_path"] = normalized.get("processed_path") or processed_path
    return normalized


def decode_real_doc(processed_path):
    raw_text = None
    if "?bytes=" in processed_path:
        base_path, bytes_param = processed_path.split("?bytes=", 1)
        stream = read_s3_object(base_path, bytes_param)
        with stream:
            raw_value = stream.read()
        if base_path.endswith(".gz"):
            raw_value = gzip.GzipFile(fileobj=io.BytesIO(raw_value)).read()
        elif base_path.endswith(".bz2"):
            raw_value = bz2.BZ2File(io.BytesIO(raw_value)).read()
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
    else:
        if processed_path.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2")):
            row = read_s3_row(processed_path)
            if row is None:
                raise ValueError(f"read_s3_row returned None for {processed_path}")
            raw_value = row.value
            raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        else:
            stream = read_s3_object(processed_path)
            with stream:
                raw_value = stream.read()
            if processed_path.endswith(".gz"):
                raw_value = gzip.GzipFile(fileobj=io.BytesIO(raw_value)).read()
            elif processed_path.endswith(".bz2"):
                raw_value = bz2.BZ2File(io.BytesIO(raw_value)).read()
            raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
    raw_text = raw_text.strip()
    if not raw_text:
        raise ValueError(f"empty content for {processed_path}")
    try:
        doc = json.loads(raw_text)
    except (TypeError, ValueError):
        if "?bytes=" in processed_path or processed_path.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2")):
            raise
        row = read_s3_row(processed_path)
        if row is None:
            raise ValueError(f"read_s3_row returned None for {processed_path}")
        raw_value = row.value
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        raw_text = raw_text.strip()
        if not raw_text:
            raise ValueError(f"empty fallback row content for {processed_path}")
        doc = json.loads(raw_text)
    return normalize_real_doc_for_chunk(doc, processed_path)


def should_scan_processed_path_by_sha(processed_path):
    text = str(processed_path or "").strip()
    return ("?bytes=" not in text) and text.endswith((".jsonl", ".jsonl.gz", ".jsonl.bz2"))


def load_real_docs_for_path(processed_path, expected_sha256s):
    expected_sha_set = {normalize_sha256(item) for item in expected_sha256s if normalize_sha256(item)}
    if not expected_sha_set:
        return {}
    if "?bytes=" in processed_path:
        doc = decode_real_doc(processed_path)
        docs_by_sha = {}
        for candidate in extract_doc_sha256_candidates(doc):
            docs_by_sha[candidate] = doc
        if not docs_by_sha and len(expected_sha_set) == 1:
            docs_by_sha[next(iter(expected_sha_set))] = doc
        return {sha: docs_by_sha[sha] for sha in docs_by_sha if sha in expected_sha_set}

    docs_by_sha = {}
    for row in read_s3_rows(processed_path, use_stream=True):
        raw_value = row.value
        raw_text = raw_value.decode("utf-8") if isinstance(raw_value, bytes) else str(raw_value)
        raw_text = raw_text.strip()
        if not raw_text:
            continue
        doc = normalize_real_doc_for_chunk(json.loads(raw_text), processed_path, getattr(row, "loc", None))
        for candidate in extract_doc_sha256_candidates(doc):
            if candidate in expected_sha_set and candidate not in docs_by_sha:
                docs_by_sha[candidate] = doc
        if len(docs_by_sha) >= len(expected_sha_set):
            break
    return docs_by_sha


def build_chunk_input_row(meta_row, real_doc):
    merged = {}
    for out_name, meta_col in METADATA_OUTPUT_FIELDS.items():
        real_value = real_doc.get(out_name)
        meta_value = meta_row.get(meta_col)
        merged[out_name] = real_value if real_value is not None else meta_value
    for name in real_field_names:
        if name in METADATA_OUTPUT_FIELDS:
            continue
        merged[name] = real_doc.get(name)
    merged["doc_loc"] = meta_row.get("doc_loc") or real_doc.get("doc_loc")
    merged["source_jsonl_path"] = meta_row.get("source_jsonl_path")
    merged["source_sha256"] = meta_row.get("source_sha256")
    merged["source_has_bytes_suffix"] = meta_row.get("source_has_bytes_suffix")
    merged["access_xinghe_repository_processed_path"] = meta_row.get("access_xinghe_repository_processed_path")
    if merged.get("processed_path") is None:
        merged["processed_path"] = meta_row.get("access_xinghe_repository_processed_path") or real_doc.get("processed_path")
    return compact_chunk_input_row(merged)


def yield_chunk_records_for_row(row, chunk_opts):
    output_row = compact_chunk_input_row(row)
    chunk_row = build_chunk_call_row(output_row)
    if "content_list" in chunk_row:
        chunk_row["content_list"] = parse_content_list_value(chunk_row.get("content_list"))
        chunk_row["content_list"] = filter_ref_text_content_list_items(chunk_row.get("content_list"))
    now_iso = utc_millis_to_iso_z(int(time.time() * 1000))
    try:
        records, _jid, _tid = chunk_plain_from_dict_row(chunk_row, chunk_opts, job_id_fallback=SPARK_JOB_ID)
    except ValueError as exc:
        error_reason = str(exc)
        if error_reason == "empty text" or error_reason.startswith("chunk split:") or error_reason.startswith("content_list 未产生任何 chunk"):
            yield build_chunk_error_record(output_row, error_reason, now_iso)
            return
        raise
    raw_primary_topic = output_row.get("primary_topic")
    primary_topic_obj = None
    if isinstance(raw_primary_topic, dict):
        primary_topic_obj = raw_primary_topic
    elif isinstance(raw_primary_topic, str):
        text = raw_primary_topic.strip()
        if text:
            try:
                parsed = json.loads(text)
                if isinstance(parsed, dict):
                    primary_topic_obj = parsed
            except (ValueError, TypeError):
                pass
    primary_topic_display_name = None
    primary_topic_subfield_display_name = None
    primary_topic_field_display_name = None
    primary_topic_domain_display_name = None
    if isinstance(primary_topic_obj, dict):
        primary_topic_display_name = primary_topic_obj.get("display_name")
        if isinstance(primary_topic_obj.get("subfield"), dict):
            primary_topic_subfield_display_name = primary_topic_obj["subfield"].get("display_name")
        if isinstance(primary_topic_obj.get("field"), dict):
            primary_topic_field_display_name = primary_topic_obj["field"].get("display_name")
        if isinstance(primary_topic_obj.get("domain"), dict):
            primary_topic_domain_display_name = primary_topic_obj["domain"].get("display_name")
    row_author = output_row.get("author")
    if isinstance(row_author, str):
        author_text = row_author.strip()
        if author_text:
            try:
                parsed_author = json.loads(author_text)
                if isinstance(parsed_author, list):
                    row_author = parsed_author
                else:
                    row_author = [row_author]
            except (ValueError, TypeError):
                row_author = [row_author]
    elif row_author is None:
        row_author = []
    elif not isinstance(row_author, list):
        row_author = [row_author]
    for r in records:
        extra_info = {}
        if isinstance(r.extra_info, (bytes, bytearray)):
            try:
                parsed_extra = json.loads(r.extra_info.decode("utf-8"))
                if isinstance(parsed_extra, dict):
                    extra_info = parsed_extra
            except (ValueError, TypeError, UnicodeDecodeError):
                extra_info = {}
        elif isinstance(r.extra_info, dict):
            extra_info = r.extra_info
        yield {
            "record_type": "chunk",
            "chunk_id": r.chunk_id,
            "doc_id": r.doc_id,
            "title": output_row.get("title") or r.title,
            "content": r.text,
            "abstract": output_row.get("abstract"),
            "url": output_row.get("url") or output_row.get("origin_url") or r.url,
            "category": r.category,
            "source_type": output_row.get("source") or r.source_type,
            "lang": r.lang,
            "job_id": r.job_id,
            "task_id": r.task_id,
            "extra_info": extra_info,
            "created_time": now_iso,
            "updated_time": now_iso,
            "offset": r.offset,
            "page_no": r.page_no,
            "chunk_no": r.chunk_no,
            "publication_published_date": output_row.get("publication_published_date"),
            "publication_published_year": output_row.get("publication_published_year"),
            "metadata_type": output_row.get("metadata_type"),
            "publication_venue_type": output_row.get("publication_venue_type"),
            "primary_topic": primary_topic_display_name,
            "primary_topic_subfield": primary_topic_subfield_display_name,
            "primary_topic_field": primary_topic_field_display_name,
            "primary_topic_domain": primary_topic_domain_display_name,
            "author": row_author,
            "publication_venue_name_unified": output_row.get("publication_venue_name_unified"),
            "access_is_oa": output_row.get("access_is_oa"),
            "citation_count": output_row.get("citation_count"),
            "influential_citation_count": output_row.get("influential_citation_count"),
        }


def chunk_only_partition(rows):
    chunk_opts = RecursiveChunkOptions(
        chunk_size=settings.ingest_chunk_size,
        chunk_overlap=settings.resolve_ingest_chunk_overlap(-1),
        keep_separator=settings.ingest_chunk_keep_separator,
    )
    for row in rows:
        yield from yield_chunk_records_for_row(row, chunk_opts)


def chunk_meta_partition(meta_rows):
    chunk_opts = RecursiveChunkOptions(
        chunk_size=settings.ingest_chunk_size,
        chunk_overlap=settings.resolve_ingest_chunk_overlap(-1),
        keep_separator=settings.ingest_chunk_keep_separator,
    )
    grouped_rows = {}
    for meta_row in meta_rows:
        item = dict(meta_row)
        normalized_sha = normalize_sha256(item.get("source_sha256"))
        if normalized_sha:
            item["source_sha256"] = normalized_sha
        processed_path = item.get("source_jsonl_path") or item.get("access_xinghe_repository_processed_path")
        if not processed_path:
            now_iso = utc_millis_to_iso_z(int(time.time() * 1000))
            yield build_chunk_error_record(item, "missing source_jsonl_path", now_iso)
            continue
        grouped_rows.setdefault(processed_path, []).append(item)

    for processed_path, path_rows in grouped_rows.items():
        expected_sha256s = [row.get("source_sha256") for row in path_rows if row.get("source_sha256")]
        try:
            docs_by_sha = load_real_docs_for_path(processed_path, expected_sha256s)
        except Exception as exc:
            now_iso = utc_millis_to_iso_z(int(time.time() * 1000))
            error_reason = f"load real doc failed: {type(exc).__name__}: {exc}"
            for row in path_rows:
                yield build_chunk_error_record(row, error_reason, now_iso)
            continue

        fallback_doc = next(iter(docs_by_sha.values())) if docs_by_sha else None
        for row in path_rows:
            source_sha256 = normalize_sha256(row.get("source_sha256"))
            real_doc = docs_by_sha.get(source_sha256)
            if real_doc is None and "?bytes=" in processed_path:
                real_doc = fallback_doc
            if real_doc is None:
                now_iso = utc_millis_to_iso_z(int(time.time() * 1000))
                yield build_chunk_error_record(
                    row,
                    f"real doc not found for processed_path={processed_path}, source_sha256={row.get('source_sha256')}",
                    now_iso,
                )
                continue
            merged_row = build_chunk_input_row(row, real_doc)
            yield from yield_chunk_records_for_row(merged_row, chunk_opts)


# ---- chunk 写出 / 断点续跑辅助 ----
def _fs_and_path(path):
    p = _Path(ensure_s3a_path(path))
    return p.getFileSystem(_hconf), p


def path_exists(path):
    fs, p = _fs_and_path(path)
    return fs.exists(p)


def delete_path(path):
    fs, p = _fs_and_path(path)
    if fs.exists(p):
        fs.delete(p, True)


def list_files_under(root_path):
    fs, root = _fs_and_path(root_path)
    if not fs.exists(root):
        return []
    result, stack = [], [root]
    while stack:
        d = stack.pop()
        for st in fs.listStatus(d):
            if st.isDirectory():
                stack.append(st.getPath())
            else:
                result.append(st.getPath().toString())
    return sorted(result)


def list_jsonl_files_under(root_path):
    return [path for path in list_files_under(root_path) if _Path(path).getName().endswith(".jsonl")]


def count_jsonl_rows_under(root_path):
    jsonl_files = list_jsonl_files_under(root_path)
    if not jsonl_files:
        return 0
    return spark.read.text([ensure_s3a_path(path) for path in jsonl_files]).count()


def list_part_files(output_dir):
    return [
        _Path(path)
        for path in list_files_under(output_dir)
        if _Path(path).getName().startswith("part-") and not _Path(path).getName().endswith(".jsonl")
    ]


def rename_part_files_to_jsonl(output_dir, max_workers=32):
    # Spark 写出的 part-* 没有 .jsonl 扩展名（下游要求 .jsonl）。每次 rename 是一次对象存储操作，
    # 串行很慢，这里用线程池并发改名。
    part_paths = list_part_files(output_dir)
    if not part_paths:
        print(f"no part files to rename under {output_dir}")
        return 0
    fs, _ = _fs_and_path(output_dir)

    def _rename(src):
        name = src.getName()
        for ext in (".json", ".txt"):
            if name.endswith(ext):
                name = name[: -len(ext)]
                break
        fs.rename(src, _Path(f"{src.getParent().toString()}/{name}.jsonl"))
        return 1

    renamed = 0
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for n in pool.map(_rename, part_paths):
            renamed += n
    print(f"renamed {renamed} part files to .jsonl under {output_dir}")
    return renamed


def build_chunk_batch_output_path(batch_index):
    return f"{CHUNK_DATA_ROOT.rstrip('/')}/batch_{int(batch_index):08d}"


def build_source_path_progress_output_path(batch_index):
    root = SOURCE_PATH_PROGRESS_ROOT.replace("s3a://", "s3://", 1).rstrip("/")
    return f"{root}/batch_{int(batch_index):08d}.jsonl"


def load_source_path_progress_df():
    progress_files = list_files_under(SOURCE_PATH_PROGRESS_ROOT)
    if not progress_files:
        return spark.createDataFrame([], SOURCE_PATH_PROGRESS_SCHEMA)
    return (
        spark.read
        .schema(SOURCE_PATH_PROGRESS_SCHEMA)
        .json([ensure_s3a_path(path) for path in progress_files])
        .select("source_jsonl_path", "batch_index", "batch_doc_count", "batch_chunk_count", "completed_at")
    )


def write_source_path_progress_batch(batch_index, batch_paths, batch_doc_count, batch_chunk_count):
    completed_at = utc_millis_to_iso_z(int(time.time() * 1000))
    rows = [
        {
            "source_jsonl_path": source_jsonl_path,
            "batch_index": int(batch_index),
            "batch_doc_count": int(batch_doc_count),
            "batch_chunk_count": int(batch_chunk_count),
            "completed_at": completed_at,
        }
        for source_jsonl_path in batch_paths
    ]
    payload = "\n".join(json.dumps(row, ensure_ascii=False, separators=(",", ":")) for row in rows) + "\n"
    put_s3_object(build_source_path_progress_output_path(batch_index), payload.encode("utf-8"))
    return completed_at


# 分批生成 chunk：按 source_jsonl_path checkpoint 断点续跑；每批写到独立 batch 目录，重跑仅覆盖当前批次。
def parse_bytes_length_from_processed_path(processed_path):
    text = str(processed_path or "").strip()
    if "?bytes=" not in text:
        return 0
    try:
        _, bytes_part = text.split("?bytes=", 1)
        _, length_str = bytes_part.split(",", 1)
        return max(0, int(length_str))
    except (TypeError, ValueError):
        return 0


def build_chunk_partition_key(meta_row, regular_bucket_count):
    path = str(meta_row.get("source_jsonl_path") or meta_row.get("access_xinghe_repository_processed_path") or "")
    byte_len = parse_bytes_length_from_processed_path(path)
    if byte_len >= int(INTRA_BATCH_OVERSIZED_BYTES_THRESHOLD):
        return f"oversized::{path}"
    if regular_bucket_count <= 1:
        return "regular::0"
    bucket = abs(hash(path)) % int(regular_bucket_count)
    return f"regular::{bucket}"


def build_batch_meta_df(batch_paths):
    batch_step_t0 = time.time()
    print(
        f"[merge][start] source_files={len(batch_paths)}, "
        f"ts={time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(batch_step_t0))}"
    )

    normalized_batch_paths = []
    normalize_t0 = time.time()
    for path in batch_paths:
        normalized_path = normalize_source_jsonl_path(path)
        if normalized_path:
            normalized_batch_paths.append(normalized_path)
    normalized_batch_paths = sorted(set(normalized_batch_paths))
    print(
        f"[merge][0/5] normalized source paths ready, raw={len(batch_paths)}, "
        f"normalized={len(normalized_batch_paths)}, elapsed={time.time() - normalize_t0:.1f}s"
    )
    if not normalized_batch_paths:
        raise RuntimeError("当前 batch 归一化后没有可读取的 source jsonl path")

    preview_t0 = time.time()
    print(f"[merge][1/5] batch_path preview = {batch_paths[:3]}")
    print(f"[merge][1/5] normalized path preview = {normalized_batch_paths[:3]}")
    print(f"[merge][1/5] preview ready, elapsed={time.time() - preview_t0:.1f}s")

    meta_load_t0 = time.time()
    batch_meta_df = load_batch_oa_metadata_df(batch_paths).persist(StorageLevel.MEMORY_AND_DISK)
    print(f"[merge][2/5] batch_meta_df loaded from StarRocks by source path (lazy), elapsed={time.time() - meta_load_t0:.1f}s")

    meta_count_t0 = time.time()
    print(f"[merge][3/5] begin batch_meta_df.count(), ts={time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(meta_count_t0))}")
    batch_meta_count = batch_meta_df.count()
    if batch_meta_count == 0:
        raise RuntimeError("当前 batch 未从 StarRocks 查到任何 metadata，请检查 source_jsonl_path 与源表是否一致")
    print(
        f"[merge][3/5] batch_meta_df materialized, rows={batch_meta_count}, "
        f"elapsed={time.time() - meta_count_t0:.1f}s"
    )

    path_type_t0 = time.time()
    metadata_by_processed_path_count = batch_meta_df.where(F.col("source_has_bytes_suffix")).count()
    metadata_by_sha_count = (
        batch_meta_df
        .where(~F.col("source_has_bytes_suffix"))
        .where(F.col("source_sha256").isNotNull() & (F.col("source_sha256") != ""))
        .count()
    )
    print(
        f"[merge][4/5] path split ready, bytes_suffix_rows={metadata_by_processed_path_count}, "
        f"sha_match_rows={metadata_by_sha_count}, elapsed={time.time() - path_type_t0:.1f}s"
    )
    print(f"[merge][5/5] metadata prep done, total elapsed={time.time() - batch_step_t0:.1f}s")
    return batch_meta_df

def run_chunk_batch(batch_paths, global_batch_index, processed_source_file_count, total_source_file_count):
    batch_output_path = build_chunk_batch_output_path(global_batch_index)
    error_output_path = f"{ERROR_OUTPUT_ROOT}/batch_{int(global_batch_index):08d}"
    batch_write_partitions = min(WRITE_PARTITIONS, max(1, len(batch_paths)))
    print(
        f"batch {global_batch_index}: source files = {len(batch_paths)}, "
        f"processed_source_files = {processed_source_file_count}/{total_source_file_count}, "
        f"write_partitions = {batch_write_partitions}"
    )

    batch_meta_df = build_batch_meta_df(batch_paths)
    batch_doc_count = batch_meta_df.select("doc_loc").distinct().count()
    chunk_input_partitions = min(batch_write_partitions, max(1, batch_doc_count))
    print(f"  chunk input partitions = {chunk_input_partitions}")

    regular_bucket_count = max(1, min(chunk_input_partitions, max(1, batch_doc_count // max(1, INTRA_BATCH_TARGET_ROWS_PER_PARTITION))))
    print(
        f"  intra-batch sharding: oversized_threshold={INTRA_BATCH_OVERSIZED_BYTES_THRESHOLD}, "
        f"regular_bucket_count={regular_bucket_count}"
    )

    chunk_records_rdd = (
        batch_meta_df
        .rdd
        .map(lambda r: r.asDict(recursive=True))
        .keyBy(lambda row: build_chunk_partition_key(row, regular_bucket_count))
        .partitionBy(max(chunk_input_partitions, regular_bucket_count))
        .values()
        .mapPartitions(chunk_meta_partition)
    )
    batch_result_json_df = spark.createDataFrame(
        chunk_records_rdd.map(lambda d: json.dumps(d, ensure_ascii=False, default=str)),
        T.StringType(),
    ).toDF("value").persist(StorageLevel.DISK_ONLY)
    batch_chunk_json_df = batch_result_json_df.where(F.get_json_object(F.col("value"), "$.record_type") == F.lit("chunk")).persist(StorageLevel.DISK_ONLY)
    batch_error_json_df = batch_result_json_df.where(F.get_json_object(F.col("value"), "$.record_type") == F.lit("error")).persist(StorageLevel.DISK_ONLY)
    batch_chunk_count = batch_chunk_json_df.count()
    batch_error_count = batch_error_json_df.count()

    (
        batch_chunk_json_df
        .repartition(batch_write_partitions)
        .write.mode(CHUNK_WRITE_MODE).text(ensure_s3a_path(batch_output_path))
    )
    rename_part_files_to_jsonl(batch_output_path)

    if batch_error_count > 0:
        (
            batch_error_json_df
            .repartition(1)
            .write.mode(CHUNK_WRITE_MODE).text(ensure_s3a_path(error_output_path))
        )
        rename_part_files_to_jsonl(error_output_path)

    batch_meta_df.unpersist()
    batch_result_json_df.unpersist()
    batch_chunk_json_df.unpersist()
    batch_error_json_df.unpersist()

    completed_at = write_source_path_progress_batch(global_batch_index, batch_paths, batch_doc_count, batch_chunk_count)
    print(
        f"  batch done: batch_index = {global_batch_index}, doc_count = {batch_doc_count}, "
        f"chunk_count = {batch_chunk_count}, error_oa_doc_count = {batch_error_count}, completed_at = {completed_at}"
    )
    return {
        "batch_doc_count": int(batch_doc_count),
        "batch_chunk_count": int(batch_chunk_count),
        "batch_error_oa_doc_count": int(batch_error_count),
    }


# 1. 构建 metadata，并按 canonical 口径筛出 OA

按每个 sha 聚合 `sha_is_oa = max(raw_is_oa)`（同一 sha 有任一行 OA 即 OA），
无 sha 的行回退本行 `raw_is_oa`，过滤 `canonical_is_oa` 得到 `oa_metadata_df`。

In [ ]:
# 防止同一 kernel 重复运行复用旧缓存
for _df_name in ("oa_metadata_df", "oa_chunk_df", "chunk_json_df", "_oa_base_df"):
    if _df_name in globals():
        try:
            globals()[_df_name].unpersist()
        except Exception:
            pass
try:
    spark.catalog.clearCache()
except Exception:
    pass

_metadata_step_t0 = time.time()
print(f"[metadata][start] {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_metadata_step_t0))}")

# 1) 取 metadata 源数据：统一用 spark.sql 从 iceberg/catalog 查询。
_load_t0 = time.time()
if spark_profile == "prod":
    metadata_source_df = load_metadata_df(limit=QUERY_LIMIT, oa_only=False, order_by_processed_path=False)
else:
    # test 用 spark.sql 小样本，保证小样本里就有 OA。
    metadata_source_df = load_metadata_df(limit=QUERY_LIMIT, oa_only=True)
print(f"[metadata][1/4] metadata_source_df ready, elapsed={time.time() - _load_t0:.1f}s")

# 2) 派生 doc_loc / source_jsonl_path / source_sha256 / raw_is_oa。
#    这里不再先对全量 metadata 做 count + persist：那会强制把大表完整物化一遍。
#
#    字段口径：
#    - doc_loc: metadata 里的 access_xinghe_repository_processed_path 原值；只作为输出字段保留，不参与真实数据定位/匹配。
#    - source_jsonl_path: 要读取的真实 jsonl 文件路径。对带/不带 ?bytes= 两类记录，都直接等于 metadata 原始 processed_path。
#      对不带 ?bytes= 的记录，这表示「先把这个文件读出来」，再在文件内按 sha256 找真实数据；它不是 join key。
#    - source_sha256: metadata 侧 sha256；先按它去重，优先最新 dt，其次优先 ?bytes= 路径，再去找真实数据。
_base_t0 = time.time()
base_metadata_df = with_metadata_dedup_columns(metadata_source_df)
print(f"[metadata][2/4] base_metadata_df built (lazy), elapsed={time.time() - _base_t0:.1f}s")

# 3) 先按 sha256 选主记录：只保留有 sha 的记录，优先最新 dt，其次优先 ?bytes= 路径。
_sha_t0 = time.time()
dedup_metadata_df = dedupe_metadata_by_sha256(base_metadata_df)
print(f"[metadata][3/4] dedup_metadata_df built (lazy), elapsed={time.time() - _sha_t0:.1f}s")

# 4) 去重后再按 OA 过滤，然后再去找真实数据。
_oa_t0 = time.time()
oa_metadata_df = (
    dedup_metadata_df.alias("m")
    .where(F.col("m.raw_is_oa"))
    .select(
        F.col("m.metadata_type").alias("metadata_type"),
        F.col("m.doi").alias("doi"),
        F.col("m.isbn13").alias("isbn13"),
        F.col("m.title").alias("title"),
        F.col("m.author").alias("author"),
        F.col("m.language").alias("language"),
        F.col("m.access_xinghe_repository_origin_url").alias("access_xinghe_repository_origin_url"),
        F.col("m.access_xinghe_repository_origin_path").alias("access_xinghe_repository_origin_path"),
        F.col("m.access_xinghe_repository_processed_path").alias("access_xinghe_repository_processed_path"),
        F.col("m.abstract").alias("abstract"),
        F.col("m.publication_published_date").alias("publication_published_date"),
        F.col("m.publication_published_year").alias("publication_published_year"),
        F.col("m.publication_venue_type").alias("publication_venue_type"),
        F.col("m.publication_venue_name_unified").alias("publication_venue_name_unified"),
        F.col("m.primary_topic").alias("primary_topic"),
        F.col("m.citation_count").alias("citation_count"),
        F.col("m.influential_citation_count").alias("influential_citation_count"),
        F.col("m.source_sha256").alias("source_sha256"),
        F.col("m.doc_loc").alias("doc_loc"),
        F.col("m.source_jsonl_path").alias("source_jsonl_path"),
        F.col("m.source_has_bytes_suffix").alias("source_has_bytes_suffix"),
        # 已按 canonical 过滤为 OA，access_is_oa 统一标 'true'。
        F.lit("true").alias("access_is_oa"),
    )
)
print(f"[metadata][4/4] oa_metadata_df built (lazy), elapsed={time.time() - _oa_t0:.1f}s")

print("metadata 源行数 (未去重)       = skipped（轻量版去掉重 count）")
print("OA doc_loc 条数 (canonical)    = skipped（轻量版去掉重 count）")
print("OA 需要读取的真实 jsonl 文件数 = skipped（轻量版去掉重 count）")
print("其中带 ?bytes= 的 OA doc 数    = skipped（轻量版去掉重 count）")
print("其中不带 ?bytes= 的 OA doc 数  = skipped（轻量版去掉重 count）")
print(f"[metadata][done] total elapsed={time.time() - _metadata_step_t0:.1f}s")
print("[metadata][note] 轻量版已去掉 metadata 阶段所有重 count；后续 chunk 阶段仍会输出待处理 batch 数与批次进度。")


# 2. 读取真实 doc 并与 metadata 合并

- 真实 doc schema 由样本 jsonl 推断（去掉 `middle_json`），再按该 schema 读取 jsonl。
- `source_jsonl_path` 表示“这条 metadata 要去读取哪个真实 jsonl 文件”；它只是**读文件路径**，不是通用 join key。
- `access_xinghe_repository_processed_path` **带 `?bytes=`** 的记录：保留原值；由于这个值已经唯一对应 metadata 对应的那条真实数据，所以不需要再在文件里按别的键查找，直接用它对上真实数据即可。
- **不带 `?bytes=`** 的记录：也不过滤；但因为它不能唯一对应某一条真实数据，所以只能在它对应的那个真实文件里，按 `sha256` 匹配真实数据。
- 整个 notebook **任何地方都不按 `doc_loc` 去定位或匹配真实数据**；`doc_loc` 只作为输出保留。
- 同名字段 `coalesce(真实 doc, metadata)` 只保留一份；其余字段全部保留。

> 注：这里一次性读取全部 OA 源文件路径。`test` 档样本小没问题；
> `prod` 全量若源文件数极大、driver 收集路径列表压力大，可参考
> `startrocks_mineru_data.ipynb` 的分批读取方式。

In [ ]:
# 1) 收集 OA 源文件路径
#    注意：source_jsonl_path 原样保留 ?bytes=，供后续按 processed_path 精确匹配；
#    但凡是“按文件读取”的地方都必须先去掉 ?bytes=，否则 spark.read.json 会把它当成路径的一部分，直接报 PATH_NOT_FOUND。
oa_source_path_df = (
    oa_metadata_df
    .select("source_jsonl_path")
    .distinct()
    .orderBy("source_jsonl_path")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
oa_source_file_count = oa_source_path_df.count()
print(f"OA source jsonl file count = {oa_source_file_count}")
if oa_source_file_count == 0:
    raise RuntimeError("oa_source_paths 为空：上一步『构建 metadata』没有筛到 OA 数据，请先检查该格输出。")


def normalize_source_jsonl_path(path):
    if path is None:
        return None
    text = str(path).strip()
    if not text:
        return None
    return text.split("?bytes=", 1)[0]


# 2) 真实 doc schema：对样本 jsonl 做一次 schema 推断（字段集合来自真实数据，
#    天然覆盖 content_list 真实结构），再去掉 middle_json 等重字段。
schema_sample_paths = []
for row in oa_source_path_df.limit(SCHEMA_SAMPLE_FILES).collect():
    normalized_path = normalize_source_jsonl_path(row["source_jsonl_path"])
    if normalized_path:
        schema_sample_paths.append(normalized_path)
schema_sample_paths = sorted(set(schema_sample_paths))
print(f"schema sample file count = {len(schema_sample_paths)}")
print(f"schema sample preview = {schema_sample_paths[:min(5, len(schema_sample_paths))]}")
real_doc_schema = infer_real_doc_schema(schema_sample_paths, exclude=EXCLUDE_REAL_FIELDS)
real_field_names = [field.name for field in real_doc_schema.fields]
real_field_set = set(real_field_names)
print(f"推断真实 doc 字段数 = {len(real_field_names)}（已排除 {sorted(EXCLUDE_REAL_FIELDS)}）")
print(f"真实 doc 字段: {real_field_names}")

# 3) 预先补齐 real doc join 所需列：
#    - processed_path：带 ?bytes= 的 metadata，直接按 access_xinghe_repository_processed_path 去对真实数据；
#    - sha256：不带 ?bytes= 的 metadata，在 source_jsonl_path 对应文件内按 sha256 去对真实数据。
real_doc_required_fields = []
for required_name, required_type in [
    ("processed_path", T.StringType()),
    ("sha256", T.StringType()),
]:
    if required_name not in real_field_set:
        real_doc_schema = real_doc_schema.add(required_name, required_type, True)
        real_field_names.append(required_name)
        real_field_set.add(required_name)
        real_doc_required_fields.append(required_name)
if real_doc_required_fields:
    print(f"补充真实 doc 必需字段: {real_doc_required_fields}")

# 4) chunk 改为按 source_jsonl_path 分批处理：
#    - 真实 doc 只读当前批次文件，避免一次性构建全量宽表；
#    - 带 ?bytes= 的 metadata 直接按 processed_path 对应那条真实数据；
#    - 不带 ?bytes= 的 metadata 在当前文件内按 sha256 对应真实数据；
#    - 整个流程不按 doc_loc 做定位/匹配；
#    - 以 source path checkpoint 做断点续跑；
#    - 每批写到独立 batch 子目录，失败重跑只覆盖当前批次，不会重复追加。
print(f"SOURCE_FILE_BATCH_SIZE={SOURCE_FILE_BATCH_SIZE}")
print(f"CHUNK_DATA_ROOT={CHUNK_DATA_ROOT}")
print(f"SOURCE_PATH_PROGRESS_ROOT={SOURCE_PATH_PROGRESS_ROOT}")


# 3. 生成 chunk

用最新的 `chunk_only_partition`（产出 dict）。把 `oa_chunk_df` 整行转 dict 喂入，显式组装 md 定义的最终 chunk 字段集合（已去掉不需要的 `query` / `entity_keys` / `embedding`）。每个 chunk 直接 `json.dumps` 成一行写出。

> 数据写到 `CHUNK_DATA_ROOT`（按 batch 子目录分桶），并在 `CHUNK_OUTPUT_ROOT` 下写 `_SUMMARY.json`。
> 写出优化：按 `source_jsonl_path` 分批读取/切分/写出；checkpoint 记录已完成批次，失败重跑只覆盖当前 batch。

In [ ]:
if OVERWRITE_EXISTING_CHUNKS:
    delete_path(CHUNK_DATA_ROOT)
    delete_path(SOURCE_PATH_PROGRESS_ROOT)
    delete_path(ERROR_OUTPUT_ROOT)

if path_exists(SOURCE_PATH_PROGRESS_ROOT):
    print("断点续跑：加载已完成的 source path checkpoint")
    source_path_progress_df = load_source_path_progress_df().persist(StorageLevel.MEMORY_AND_DISK)
    completed_source_paths_df = (
        source_path_progress_df
        .select("source_jsonl_path")
        .dropDuplicates(["source_jsonl_path"])
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    completed_source_file_count = completed_source_paths_df.count()
    completed_batch_index = int(
        source_path_progress_df
        .agg(F.max(F.coalesce(F.col("batch_index"), F.lit(0))).alias("max_batch_index"))
        .collect()[0]["max_batch_index"]
        or 0
    )
    completed_doc_count = int(
        source_path_progress_df
        .groupBy("batch_index")
        .agg(F.max(F.coalesce(F.col("batch_doc_count"), F.lit(0))).alias("batch_doc_count"))
        .agg(F.sum("batch_doc_count").alias("doc_count"))
        .collect()[0]["doc_count"]
        or 0
    )
    completed_chunk_count = int(
        source_path_progress_df
        .groupBy("batch_index")
        .agg(F.max(F.coalesce(F.col("batch_chunk_count"), F.lit(0))).alias("batch_chunk_count"))
        .agg(F.sum("batch_chunk_count" ).alias("chunk_count"))
        .collect()[0]["chunk_count"]
        or 0
    )
    source_path_progress_df.unpersist()
else:
    completed_source_paths_df = spark.createDataFrame(
        [],
        T.StructType([T.StructField("source_jsonl_path", T.StringType(), True)]),
    ).persist(StorageLevel.MEMORY_AND_DISK)
    completed_source_file_count = 0
    completed_batch_index = 0
    completed_doc_count = 0
    completed_chunk_count = 0

pending_source_paths_df = (
    oa_source_path_df
    .join(completed_source_paths_df, "source_jsonl_path", "left_anti")
    .orderBy("source_jsonl_path")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
pending_source_file_count = pending_source_paths_df.count()
pending_batch_count = (pending_source_file_count + SOURCE_FILE_BATCH_SIZE - 1) // SOURCE_FILE_BATCH_SIZE
print(f"source file 总数 = {oa_source_file_count}")
print(f"已完成 checkpoint 文件数 = {completed_source_file_count}")
print(f"已完成 batch 数 = {completed_batch_index}")
print(f"已完成 oa doc 数 = {completed_doc_count}")
print(f"已完成 chunk 数 = {completed_chunk_count}")
print(f"待处理 source file 数 = {pending_source_file_count}")
print(f"待处理 batch 数 = {pending_batch_count}")

processed_source_file_count = completed_source_file_count
total_chunk_count = completed_chunk_count
total_oa_doc_count = completed_doc_count
total_error_oa_doc_count = 0
batch_paths = []
global_batch_index = completed_batch_index

for row in pending_source_paths_df.toLocalIterator():
    batch_paths.append(row["source_jsonl_path"])
    if len(batch_paths) < SOURCE_FILE_BATCH_SIZE:
        continue

    global_batch_index += 1
    processed_source_file_count += len(batch_paths)
    batch_result = run_chunk_batch(
        batch_paths,
        global_batch_index,
        processed_source_file_count,
        oa_source_file_count,
    )
    total_chunk_count += batch_result["batch_chunk_count"]
    total_oa_doc_count += batch_result["batch_doc_count"]
    total_error_oa_doc_count += batch_result["batch_error_oa_doc_count"]
    batch_paths = []

if batch_paths:
    global_batch_index += 1
    processed_source_file_count += len(batch_paths)
    batch_result = run_chunk_batch(
        batch_paths,
        global_batch_index,
        processed_source_file_count,
        oa_source_file_count,
    )
    total_chunk_count += batch_result["batch_chunk_count"]
    total_oa_doc_count += batch_result["batch_doc_count"]
    total_error_oa_doc_count += batch_result["batch_error_oa_doc_count"]

# 重新加载 checkpoint，并基于落盘结果汇总最新 summary。
source_path_progress_df = load_source_path_progress_df().persist(StorageLevel.MEMORY_AND_DISK)
materialized_progress_row_count = source_path_progress_df.count()
materialized_source_file_count = source_path_progress_df.select("source_jsonl_path").dropDuplicates(["source_jsonl_path"]).count()
materialized_batch_count = source_path_progress_df.select("batch_index").dropDuplicates(["batch_index"]).count()
materialized_doc_count = int(
    source_path_progress_df
    .groupBy("batch_index")
    .agg(F.max(F.coalesce(F.col("batch_doc_count"), F.lit(0))).alias("batch_doc_count"))
    .agg(F.sum("batch_doc_count").alias("doc_count"))
    .collect()[0]["doc_count"]
    or 0
)
materialized_chunk_count = int(
    source_path_progress_df
    .groupBy("batch_index")
    .agg(F.max(F.coalesce(F.col("batch_chunk_count"), F.lit(0))).alias("batch_chunk_count"))
    .agg(F.sum("batch_chunk_count").alias("chunk_count"))
    .collect()[0]["chunk_count"]
    or 0
)
source_path_progress_df.unpersist()
completed_source_paths_df.unpersist()
pending_source_paths_df.unpersist()

oa_doc_count = int(materialized_doc_count)
error_oa_doc_count = int(count_jsonl_rows_under(ERROR_OUTPUT_ROOT))
success_oa_doc_count = int(max(0, oa_doc_count - error_oa_doc_count))
if error_oa_doc_count > oa_doc_count:
    raise RuntimeError(
        f"error_oa_doc_count({error_oa_doc_count}) > oa_doc_count({oa_doc_count})，请检查 error 目录是否存在重复或脏数据"
    )

summary_payload = {
    "output_root": CHUNK_OUTPUT_ROOT,
    "data_root": CHUNK_DATA_ROOT,
    "oa_doc_count": oa_doc_count,
    "success_oa_doc_count": success_oa_doc_count,
    "error_oa_doc_count": error_oa_doc_count,
    "completed_source_file_count": int(materialized_source_file_count),
    "batch_count": int(materialized_batch_count),
    "chunk_count": int(materialized_chunk_count),
}
put_s3_object(
    f"{CHUNK_OUTPUT_ROOT.rstrip('/')}/_SUMMARY.json",
    json.dumps(summary_payload, ensure_ascii=False, indent=2).encode("utf-8"),
)

oa_source_path_df.unpersist()

print(f"materialized progress row count = {materialized_progress_row_count}")
print(f"materialized source file count = {materialized_source_file_count}")
print(f"materialized batch count = {materialized_batch_count}")
print(f"materialized oa doc count = {materialized_doc_count}")
print(f"materialized chunk count = {materialized_chunk_count}")
print(f"runtime error oa doc count = {total_error_oa_doc_count}")
print(f"materialized error oa doc count = {error_oa_doc_count}")
print(f"success oa doc count = {success_oa_doc_count}")
print(f"oa doc count check: {success_oa_doc_count} + {error_oa_doc_count} = {success_oa_doc_count + error_oa_doc_count} (expected {oa_doc_count})")
print(f"chunk 写出完成 -> {CHUNK_OUTPUT_ROOT}")
print(f"  chunk 数据(.jsonl): {CHUNK_DATA_ROOT}/ ; error 数据(.jsonl): {ERROR_OUTPUT_ROOT}/ ; 汇总: {CHUNK_OUTPUT_ROOT}/_SUMMARY.json")

